In [ ]:
pip install pandas pulp #libreria per ottimizzazione lineare intera

In [ ]:
import pandas as pd
import pulp

In [ ]:
days = ["Sun", "Mon", "Tue", "Wed", "Thu", "Fri", "Sat"]

# Infermiere richieste per giorno
demand = {
    "Sun": 5,
    "Mon": 4,
    "Tue": 3,
    "Wed": 3,
    "Thu": 3,
    "Fri": 4,
    "Sat": 6
}

# Costo per infermiera-giorno
cost = {
    "Sun": 320,
    "Mon": 240,
    "Tue": 240,
    "Wed": 240,
    "Thu": 240,
    "Fri": 240,
    "Sat": 320
}

# 7 possibili schedulazioni (1 = lavora, 0 = riposo)
schedules = {
    "S1": {"Sun":0,"Mon":0,"Tue":1,"Wed":1,"Thu":1,"Fri":1,"Sat":1},
    "S2": {"Sun":1,"Mon":0,"Tue":0,"Wed":1,"Thu":1,"Fri":1,"Sat":1},
    "S3": {"Sun":1,"Mon":1,"Tue":0,"Wed":0,"Thu":1,"Fri":1,"Sat":1},
    "S4": {"Sun":1,"Mon":1,"Tue":1,"Wed":0,"Thu":0,"Fri":1,"Sat":1},
    "S5": {"Sun":1,"Mon":1,"Tue":1,"Wed":1,"Thu":0,"Fri":0,"Sat":1},
    "S6": {"Sun":1,"Mon":1,"Tue":1,"Wed":1,"Thu":1,"Fri":0,"Sat":0},
    "S7": {"Sun":0,"Mon":1,"Tue":1,"Wed":1,"Thu":1,"Fri":1,"Sat":0}
}

In [ ]:
model1 = pulp.LpProblem("Nurse_Staffing", pulp.LpMinimize)

In [ ]:
# Variabili: numero di infermiere per turno
x = pulp.LpVariable.dicts("Nurses", schedules.keys(), lowBound=0, cat="Integer")

In [ ]:
# Funzione obiettivo: costo totale
model1 += pulp.lpSum(
    x[s] * sum(cost[d] * schedules[s][d] for d in days)
    for s in schedules
)
#se l’infermiera lavora quel giorno (schedules[s][d] = 1), aggiunge il costo di quel giorno
#Se non lavora (schedules[s][d] = 0), non aggiunge nulla.
#il valore viene ripetuto per il numero di infermire x[s] assegnate a quella schedule
#Infine si somma per tutte le schedule per trovare il totale


In [ ]:
# Vincoli di copertura giornaliera
for d in days:
    model1 += pulp.lpSum(x[s] * schedules[s][d] for s in schedules) >= demand[d]
#x[s] * schedules[s][d] = numero di infermiere di quel turno che lavorano quel giorno.

In [ ]:
# Risoluzione
model1.solve()

1

In [ ]:
# Numero di infermiere per schedule
print("FASE 1 - Infermiere per turno")
required_per_schedule = {}
for s in schedules:
    required_per_schedule[s] = int(x[s].value())
    print(s, ":", required_per_schedule[s])

FASE 1 - Infermiere per turno
S1 : 1
S2 : 1
S3 : 2
S4 : 1
S5 : 1
S6 : 0
S7 : 0


In [ ]:
# Numero di infermiere per giorno
print("\nInfermiere totali per giorno:")
total_per_day = {}
for d in days:
    total_per_day[d] = sum(required_per_schedule[s] * schedules[s][d] for s in schedules)
    print(f"{d}: {total_per_day[d]} (richieste: {demand[d]})")




Infermiere totali per giorno:
Sun: 5 (richieste: 5)
Mon: 4 (richieste: 4)
Tue: 3 (richieste: 3)
Wed: 3 (richieste: 3)
Thu: 4 (richieste: 3)
Fri: 5 (richieste: 4)
Sat: 6 (richieste: 6)


In [ ]:
# Costo totale
total_cost = pulp.value(model1.objective)
print(f"\nCosto totale settimanale: ${total_cost}")


Costo totale settimanale: $8080.0


In [ ]:
#creiamo un dizionario con le infermiere e i rispettivi anni di anzianità
employees = {
    "Mary": 5,
    "Anne": 6,
    "Susan": 4,
    "Tom": 7,
    "Cathy": 3,
    "Jane": 2
}

max_years = max(employees.values())

seniority_factor = {e: round(employees[e]/max_years,2) for e in employees}

print(seniority_factor)

{'Mary': 0.71, 'Anne': 0.86, 'Susan': 0.57, 'Tom': 1.0, 'Cathy': 0.43, 'Jane': 0.29}


In [ ]:
# Preferenze (A–E)
preferences = {
    "Mary":  {"S1":3,"S2":1,"S3":2,"S4":4,"S5":5},
    "Anne":  {"S1":3,"S2":2,"S3":1,"S4":4,"S5":5},
    "Susan": {"S1":2,"S2":3,"S3":5,"S4":1,"S5":4},
    "Tom":   {"S1":3,"S2":2,"S3":4,"S4":1,"S5":5},
    "Cathy": {"S1":3,"S2":4,"S3":5,"S4":2,"S5":1},
    "Jane":  {"S1":5,"S2":4,"S3":3,"S4":2,"S5":1}
}

In [ ]:
# Numero richiesto di infermiere per turno (output fase 1)
required_per_schedule = {
    "S1": 1,
    "S2": 1,
    "S3": 2,
    "S4": 1,
    "S5": 1
}


In [ ]:
model2 = pulp.LpProblem("Schedule_Assignment", pulp.LpMaximize)

In [ ]:
y = pulp.LpVariable.dicts(
    "Assign",
    [(e, s) for e in employees.keys() for s in preferences[e]],
    cat="Binary"
)
#y[(e, s)] = 1 se l’infermiera e è assegnata al turno s, 0 altrimenti

In [ ]:
# Funzione obiettivo: massimizzare WPS
model2 += pulp.lpSum(
    y[(e, s)] * preferences[e][s] * seniority_factor[e]
    for e in employees for s in preferences[e]
)
#WPS = y[(e,s)] * preferenza infermiera * seniority factor

In [ ]:
# Ogni infermiera → massimo un turno
for e in employees:
    model2 += pulp.lpSum(y[(e, s)] for s in preferences[e]) == 1
# for e in employees controlla su tutte le infermiere
# for s in preferences[e] controlla su tutti i turni disponibili per quell’infermiera

In [ ]:
turns = list(required_per_schedule.keys())  # prendi solo i turni da assegnare

for s in turns:
    model2 += pulp.lpSum(y[(e, s)] for e in employees.keys()) == required_per_schedule.get(s, 0)
#somma delle infermiere assegnate a un turno = numero richiesto da Fase 1

In [ ]:
# Risoluzione
model2.solve()



1

In [ ]:
# Output
print("\nFASE 2 - Assegnazioni")
for e in employees:
    for s in preferences[e]:
        if y[(e, s)].value() == 1:
            print(e, "assegnata a", s)


FASE 2 - Assegnazioni
Mary assegnata a S1
Anne assegnata a S4
Susan assegnata a S3
Tom assegnata a S5
Cathy assegnata a S2
Jane assegnata a S1


In [ ]:
wps = 0
print("Assegnazioni infermiere-turno:")
for e in employees:
    for s in preferences[e]:
        if y[(e, s)].value() == 1:
            score = preferences[e][s] * seniority_factor[e]
            wps += score
            print(f"{e} assegnata a {s}, punteggio ponderato: {score:.2f}")

print(f"\nWeighted Preference Score totale (WPS): {wps:.2f}")

Assegnazioni infermiere-turno:
Mary assegnata a S1, punteggio ponderato: 2.13
Anne assegnata a S4, punteggio ponderato: 3.44
Susan assegnata a S3, punteggio ponderato: 2.85
Tom assegnata a S5, punteggio ponderato: 5.00
Cathy assegnata a S2, punteggio ponderato: 1.72
Jane assegnata a S1, punteggio ponderato: 1.45

Weighted Preference Score totale (WPS): 16.59
